In [1]:
pip install simpy

In [2]:
import simpy
import numpy as np
import pandas as pd
import random

In [5]:
def customer(env, servers, service_time, wait_times):
    arrival = env.now
    with servers.request() as request:
        yield request
        wait = env.now - arrival
        wait_times.append(wait)
        yield env.timeout(service_time)

In [3]:
def run_simulation(arrival_rate, service_rate, num_servers, sim_time):

    env = simpy.Environment()
    servers = simpy.Resource(env, capacity=num_servers)

    wait_times = []

    def arrival_process(env):
        while True:
            yield env.timeout(random.expovariate(arrival_rate))
            env.process(customer(env, servers,
                                 random.expovariate(service_rate),
                                 wait_times))

    env.process(arrival_process(env))
    env.run(until=sim_time)

    return np.mean(wait_times) if wait_times else 0

In [6]:
#Generate 1000 Simulations
data = []

for i in range(1000):

    arrival = random.uniform(1,10)
    service = random.uniform(2,15)
    servers = random.randint(1,5)
    sim_time = random.randint(50,500)

    avg_wait = run_simulation(arrival,service,servers,sim_time)

    data.append([arrival,service,servers,sim_time,avg_wait])

df = pd.DataFrame(data,columns=[
'ArrivalRate','ServiceRate','Servers','SimTime','AvgWait'
])

df.head()

,ArrivalRate,ServiceRate,Servers,SimTime,AvgWait
0,7.194930,10.414275,3,77,0.001667
1,1.671283,11.338847,2,418,0.000614
2,6.291728,4.336366,5,283,0.001564
3,5.412911,5.577398,5,373,0.000053
4,5.093122,13.352133,4,264,0.000028


ML Model Comparison

In [7]:
from sklearn.model_selection import train_test_split

X = df.iloc[:,:-1]
y = df.iloc[:,-1]

X_train,X_test,y_train,y_test = train_test_split(
X,y,test_size=0.2,random_state=42)

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_squared_error

In [9]:
models = {
"Linear":LinearRegression(),
"DecisionTree":DecisionTreeRegressor(),
"RandomForest":RandomForestRegressor(),
"SVR":SVR(),
"KNN":KNeighborsRegressor(),
"GradientBoost":GradientBoostingRegressor()
}

results=[]

for name,model in models.items():

    model.fit(X_train,y_train)
    pred=model.predict(X_test)

    r2=r2_score(y_test,pred)
    rmse=np.sqrt(mean_squared_error(y_test,pred))

    results.append([name,r2,rmse])

compare=pd.DataFrame(results,
columns=["Model","R2 Score","RMSE"])

compare

,Model,R2 Score,RMSE
0,Linear,0.206563,15.631557
1,DecisionTree,0.912572,5.188864
2,RandomForest,0.929941,4.644929
3,SVR,-0.061996,18.084540
4,KNN,-0.244374,19.575887
5,GradientBoost,0.848086,6.839820
